# Text Preprocessing and Sentiment Analysis

This is a cleaned, notebook-friendly version of the supplied sentiment program. It tokenizes text, removes punctuation and stop words, applies stemming, and uses VADER on the original sentence so negation is preserved.

## Anaconda / Jupyter Notebook
Run the cells from top to bottom. If a required package is missing, run `%pip install package-name` in a notebook cell, then restart the kernel if needed.


In [ ]:
%pip install nltk


In [ ]:
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.sentiment import SentimentIntensityAnalyzer

# Download resources required by NLTK
for resource in ["punkt", "punkt_tab", "stopwords", "vader_lexicon"]:
    nltk.download(resource, quiet=True)

texts = [
    "The movie was excellent and I really enjoyed it.",
    "The product is very bad and disappointing.",
    "The service was good and the staff were helpful.",
    "I do not like this product because it is terrible.",
    "The movie was okay and the story was average."
]

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
sia = SentimentIntensityAnalyzer()


In [ ]:
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]
    return tokens, filtered_tokens, stemmed_tokens

results = []
for text in texts:
    tokens, filtered, stemmed = preprocess_text(text)

    # Analyze original text to preserve negation such as "not good"
    scores = sia.polarity_scores(text)
    compound = scores["compound"]

    if compound >= 0.05:
        sentiment = "Positive"
    elif compound <= -0.05:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    results.append({
        "Original Text": text,
        "Tokens": tokens,
        "After Stop-word Removal": filtered,
        "After Stemming": stemmed,
        "Positive Score": scores["pos"],
        "Negative Score": scores["neg"],
        "Neutral Score": scores["neu"],
        "Compound Score": compound,
        "Sentiment": sentiment
    })

df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", 100)
display(df)


In [ ]:
print("Sentiment summary:")
print(df["Sentiment"].value_counts())
print("\nPercentage of each sentiment:")
print((df["Sentiment"].value_counts(normalize=True) * 100).round(2).astype(str) + "%")

df["Sentiment"].value_counts().plot(kind="bar", figsize=(7, 5), title="Sentiment Analysis Results")
plt.xlabel("Sentiment")
plt.ylabel("Number of Texts")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
